In [ ]:
# TGraph records carry attributes; convert to geometry only when needed.
def _as_topology(value):
    from topologicpy.Topology import Topology
    from topologicpy.Vertex import Vertex
    if isinstance(value, (list, tuple)):
        return [_as_topology(item) for item in value]
    if not isinstance(value, dict) or "index" not in value or "dictionary" not in value:
        return value
    representation = value.get("representation")
    attributes = value["dictionary"]
    if representation is None and "src" not in value:
        representation = Vertex.ByCoordinates(
            attributes.get("x", 0), attributes.get("y", 0), attributes.get("z", 0))
    if representation is None:
        raise ValueError("This graph record has no geometry representation")
    return Topology.SetDictionary(representation, attributes)


def _record_dictionary(value):
    from topologicpy.Topology import Topology
    if isinstance(value, dict) and "dictionary" in value:
        return value["dictionary"]
    return Topology.Dictionary(value)


def _path_wire(graph, indices):
    from topologicpy.TGraph import TGraph
    from topologicpy.Vertex import Vertex
    from topologicpy.Wire import Wire
    if not indices or len(indices) < 2:
        return None
    vertices = [Vertex.ByCoordinates(TGraph.Coordinates(graph, i)) for i in indices]
    return Wire.ByVertices(vertices, close=False)


def _edge_records(graph):
    from topologicpy.TGraph import TGraph
    records = TGraph.Edges(graph)
    if any(record.get("representation") is None for record in records):
        geometry = TGraph.Edges(graph, asTopologic=True, segmentCurves=False)
        if len(geometry) != len(records):
            raise ValueError("Could not generate one geometry representation per graph edge")
        for record, edge in zip(records, geometry):
            record["representation"] = edge
    return records


# Tutorial on how to route paths using edge capacities
The idea here is that each edge has an assigned maximum capacity. When trying to find shortest paths for multiple pipes/cables etc, the solution repsects the capacity and also reduces it if it can be occupied.

In [ ]:
# This cell is not needed if you have pip installed topologicpy
import sys
sys.path.append("C:/Users/sarwj/OneDrive - Cardiff University/Documents/GitHub/topologicpy/src")

## Import the needed topologicpy libraries

In [ ]:
from topologicpy.Vertex import Vertex
from topologicpy.Wire import Wire
from topologicpy.Face import Face
from topologicpy.Shell import Shell
from topologicpy.Cluster import Cluster
from topologicpy.Topology import Topology
from topologicpy.Dictionary import Dictionary
from topologicpy.TGraph import TGraph


## Import an example DXF file to create the base graph. Look for and download "network.dxf" on github and modify the path accordingly

In [ ]:
topologies = Topology.ByDXFPath(path=r"C:\Users\sarwj\OneDrive - Cardiff University\Desktop\network.dxf",
                                silent=True)

## Main loop. Study the comments in the code

In [ ]:
from topologicpy.Helper import Helper
import random
import math
# Get and flatten all the edges from the imported DXF file
edges = [Topology.Edges(_as_topology(w)) for w in topologies]
edges = Helper.Flatten(edges)
cluster = Topology.SelfMerge(Cluster.ByTopologies(_as_topology(edges)))

# Get all the vertices from the cluster (de-duplicated)
vertices = Topology.Vertices(_as_topology(cluster))

# Build a graph from the vertices and edges
g = TGraph.ByVerticesEdges(vertices, edges)

# The graph creates its own vertices and edges so get those back.
vertices = TGraph.Vertices(g)
edges = _edge_records(g)

 # Start and End vertices are hard-wired to be top-left and bottom-right of network.dxf graph.
 # Change to your own use case if you changed the DXF file.
sv = Vertex.ByCoordinates(_as_topology([2.0, 3.000353, 0.0]))
ev = Vertex.ByCoordinates(_as_topology([9.0, 0.000353, 0.0]))

# Centers is just a list of edge centers to hold and display the capacities in Plotly.
centers = []
for e in edges:
    capacity = math.floor(random.uniform(2,6)) # Assign a random capacity between 2 and 6
    d = Dictionary.ByKeyValue("capacity", capacity)
    e = Topology.SetDictionary(e, d) # The edge now has a capacity
    c = Topology.Centroid(_as_topology(e))
    d = Dictionary.SetValueAtKey(d, "size", 0.0001) # I do not want to see the centers, just their label.
    d = Dictionary.SetValueAtKey(d, "label", str(capacity))
    c = Topology.SetDictionary(c, d)
    centers.append(c)

# This function tells shortest path if an edge can be traversed or not.
# An edge cannot be traversed if its capacity is <= 0.
def edge_filter(edge):
    d = _record_dictionary(edge)
    capacity = Dictionary.ValueAtKey(d, "capacity")
    if capacity <= 0:
        return False
    return True

sp_list = [] # This is the list to hold all the shortest paths you are creating.
n_paths = 7 # This is how many paths you want to route along the graph network.
# In this scenario, all paths start from the same location and end at the same location.
for i in range(n_paths):
    return_values = TGraph.ShortestPath(g,
                                       sv,
                                       ev,
                                       turnWeight=0.3,
                                       edgeFilter=edge_filter,
                                       returnVertices=True,
                                       returnEdges=True)
    if return_values: # A shortest path may not be found, so check that this is not None.
        # TGraph returns vertex indices and the requested records.
        # vi_list is the list of vertex indices of the shortest path
        # so you can find them in the original list of graph vertices
        # ei_list is the list of edge indices so you 
        # can find them in the original list of graph edges.
        vi_list, path_vertices, path_edges = return_values
        sp = _path_wire(g, vi_list)
        ei_list = [edge["index"] for edge in path_edges] 
        sp = Topology.Translate(_as_topology(sp), 0, 0, (i + 1) * 0.2)
        sp_list.append(sp)
        # Reduce capacities
        for i in ei_list:
            e = edges[i]
            d = _record_dictionary(e)
            capacity = Dictionary.ValueAtKey(d, "capacity")
            if capacity > 0:
                capacity = capacity - 1 # Here is where we reduce the capacity by one.
            d = Dictionary.SetValueAtKey(d, key="capacity", value=capacity)
            e = Topology.SetDictionary(e, d) # Now the edge has a reduced capacity

for i, s_p in enumerate(sp_list):
    print(i + 1, 'Length:', round(Wire.Length(_as_topology(s_p)), 2), 'm')
Topology.Show(g, centers, sp_list, 
              vertexLabelKey="label", vertexSizeKey="size",
              showVertexLabel=True,
              backgroundColor="beige",
              renderer="browser",
              width=1290,
              height=730)

In [ ]:
@staticmethod
def GoldenSpiral(width: float = 1.0,
                 numSides: int = 96,
                 origin=None,
                 placement: str = "center",
                 direction: list = [0, 0, 1],
                 clockwise: bool = False,
                 squareIterations: int = 10,
                 edgeTypeKey: str = "edge_type",
                 squareEdgeValue: str = "square",
                 spiralEdgeValue: str = "spiral",
                 mantissa: int = 6,
                 tolerance: float = 0.0001,
                 silent: bool = False):
    """
    Recursive golden-rectangle subdivision + classic quarter-circle “golden spiral”.

    Output contains:
      - edges of all subdivision squares (tagged),
      - quarter-circle arcs (approximated by polylines) inside each square (tagged),
        with `numSides` distributed across all arcs.

    Parameters
    ----------
    width : float
        Long side of the outer golden rectangle (height is width/phi).
    numSides : int
        Total number of segments for the WHOLE spiral (distributed across arcs).
    origin : topologic_core.Vertex, optional
    placement : str
        "center", "lowerleft", "lowerright", "upperleft", "upperright"
    direction : list
        Plane normal (like Wire.Rectangle).
    clockwise : bool
        Spiral direction.
    squareIterations : int
        Number of recursion steps (squares + arcs).
    edgeTypeKey, squareEdgeValue, spiralEdgeValue : tagging controls.

    Returns
    -------
    topologic_core.Wire
    """

    from topologicpy.Vertex import Vertex
    from topologicpy.Edge import Edge
    from topologicpy.Wire import Wire
    from topologicpy.Dictionary import Dictionary
    from topologicpy.Topology import Topology

    import math

    # -----------------------------
    # Helpers
    # -----------------------------
    def _safe_vertex(v):
        return v if v is not None else Vertex.Origin()

    def _round(x):
        return round(float(x), int(mantissa))

    def _vec_norm(v):
        l = math.sqrt(v[0]*v[0] + v[1]*v[1] + v[2]*v[2])
        if l < 1e-12:
            return [0.0, 0.0, 1.0]
        return [v[0]/l, v[1]/l, v[2]/l]

    def _vec_cross(a, b):
        return [a[1]*b[2]-a[2]*b[1], a[2]*b[0]-a[0]*b[2], a[0]*b[1]-a[1]*b[0]]

    def _vec_dot(a, b):
        return a[0]*b[0] + a[1]*b[1] + a[2]*b[2]

    def _edge_with_tag(v0, v1, tag_value):
        e = Edge.ByStartVertexEndVertex(_as_topology(v0), _as_topology(v1), tolerance=tolerance)
        d = Dictionary.ByKeysValues([edgeTypeKey], [tag_value])
        return Topology.SetDictionary(e, d)

    def _square_edges(sx, sy, s):
        bl = Vertex.ByCoordinates(_as_topology(_round(sx)), _as_topology(_round(sy)), 0.0)
        br = Vertex.ByCoordinates(_as_topology(_round(sx + s)), _as_topology(_round(sy)), 0.0)
        tr = Vertex.ByCoordinates(_as_topology(_round(sx + s)), _as_topology(_round(sy + s)), 0.0)
        tl = Vertex.ByCoordinates(_as_topology(_round(sx)), _as_topology(_round(sy + s)), 0.0)
        return [
            _edge_with_tag(bl, br, squareEdgeValue),
            _edge_with_tag(br, tr, squareEdgeValue),
            _edge_with_tag(tr, tl, squareEdgeValue),
            _edge_with_tag(tl, bl, squareEdgeValue)
        ], (bl, br, tr, tl)

    def _arc_edges_from_square_corners(corners, k, nseg):
        """
        k = 0,1,2,3 corresponds to peel-side cycle (CCW):
          0: left   -> center=bl, arc tl->br
          1: bottom -> center=br, arc bl->tr
          2: right  -> center=tr, arc br->tl
          3: top    -> center=tl, arc tr->bl

        For clockwise: reverse traversal by swapping (ang0, ang1).
        """
        bl, br, tr, tl = corners
        if k == 0:
            center = bl
            ang0, ang1 = math.radians(90),  math.radians(0)     # tl -> br
            r = _dist(center, br)
        elif k == 1:
            center = br
            ang0, ang1 = math.radians(180), math.radians(90)    # bl -> tr
            r = _dist(center, tr)
        elif k == 2:
            center = tr
            ang0, ang1 = math.radians(270), math.radians(180)   # br -> tl
            r = _dist(center, tl)
        else:
            center = tl
            ang0, ang1 = math.radians(0),   math.radians(-90)   # tr -> bl
            r = _dist(center, bl)

        if clockwise:
            ang0, ang1 = ang1, ang0

        if nseg < 1:
            nseg = 1

        pts = []
        cx, cy = Vertex.X(_as_topology(center)), Vertex.Y(_as_topology(center))
        for i in range(nseg + 1):
            t = float(i) / float(nseg)
            a = ang0 + t * (ang1 - ang0)
            x = cx + r * math.cos(a)
            y = cy + r * math.sin(a)
            pts.append(Vertex.ByCoordinates(_as_topology(_round(x)), _as_topology(_round(y)), 0.0))

        edges = []
        for i in range(len(pts) - 1):
            edges.append(_edge_with_tag(pts[i], pts[i+1],ison(pts[i+1],pts[i], spiralEdgeValue)))
        return edges

    def _dist(a, b):
        dx = Vertex.X(_as_topology(a)) - Vertex.X(_as_topology(b))
        dy = Vertex.Y(_as_topology(a)) - Vertex.Y(_as_topology(b))
        return math.sqrt(dx*dx + dy*dy)

    # NOTE: tiny typo guard: ensure we call _edge_with_tag
    def Roca(a,b,tag):
        return _edge_with_tag(a,b,tag)

    def _arc_edges_from_square_corners(corners, k, nseg):
        """
        Corners = (bl, br, tr, tl)

        For each square, the quarter-circle endpoints lie on the square diagonal (opposite corners),
        and the arc center is one of the *other* two corners. “Flip along the diagonal” means:
        swap the center to the other corner on that diagonal’s perpendicular pair.

        k (subdivision step index) cycles 0,1,2,3 and determines which diagonal we use:
        k=0: endpoints (tl, br), centers swap: bl <-> tr
        k=1: endpoints (bl, tr), centers swap: br <-> tl
        k=2: endpoints (br, tl), centers swap: tr <-> bl
        k=3: endpoints (tr, bl), centers swap: tl <-> br

        Arc is sampled into `nseg` edges and tagged as spiral edges.
        If `clockwise` is True, the arc direction is reversed (swap angle start/end).
        """
        bl, br, tr, tl = corners

        # --- choose diagonal endpoints and the FLIPPED center ---
        if k == 0:
            p_start, p_end = tl, br
            center = tr          # flipped (was bl)
        elif k == 1:
            p_start, p_end = bl, tr
            center = tl          # flipped (was br)
        elif k == 2:
            p_start, p_end = br, tl
            center = bl          # flipped (was tr)
        else:  # k == 3
            p_start, p_end = tr, bl
            center = br          # flipped (was tl)

        # radius is square size
        cx, cy = Vertex.X(_as_topology(center)), Vertex.Y(_as_topology(center))
        r = _dist(center, p_start)

        # compute start/end angles from the actual points
        ang0 = math.atan2(Vertex.Y(_as_topology(p_start)) - cy, Vertex.X(_as_topology(p_start)) - cx)
        ang_end = math.atan2(Vertex.Y(_as_topology(p_end)) - cy, Vertex.X(_as_topology(p_end)) - cx)

        # We want a quarter turn (±pi/2). Pick the candidate that matches the actual end angle.
        cand1 = ang0 + math.pi/2.0
        cand2 = ang0 - math.pi/2.0

        # normalize angular difference to [-pi, pi]
        def _adiff(a, b):
            d = a - b
            while d <= -math.pi:
                d += 2.0*math.pi
            while d > math.pi:
                d -= 2.0*math.pi
            return d

        # choose candidate end angle closest to the actual end angle modulo 2pi
        if abs(_adiff(cand1, ang_end)) <= abs(_adiff(cand2, ang_end)):
            ang1 = cand1
        else:
            ang1 = cand2

        # reverse traversal if clockwise
        if clockwise:
            ang0, ang1 = ang1, ang0

        nseg = max(1, int(nseg))

        pts = []
        for i in range(nseg + 1):
            t = float(i) / float(nseg)
            a = ang0 + t * (ang1 - ang0)
            x = cx + r * math.cos(a)
            y = cy + r * math.sin(a)
            pts.append(Vertex.ByCoordinates(_as_topology(_round(x)), _as_topology(_round(y)), 0.0))

        edges = []
        for i in range(len(pts) - 1):
            edges.append(_edge_with_tag(pts[i], pts[i+1], spiralEdgeValue))
        return edges

    # -----------------------------
    # Validate
    # -----------------------------
    try:
        width = float(width)
        if width <= 0:
            if not silent:
                print("Wire.GoldenSpiral: width must be > 0.")
            return None
        numSides = max(4, int(numSides))
        squareIterations = max(1, int(squareIterations))
    except Exception:
        if not silent:
            print("Wire.GoldenSpiral: invalid inputs.")
        return None

    # -----------------------------
    # Outer golden rectangle (local XY)
    # -----------------------------
    phi = (1.0 + math.sqrt(5.0)) / 2.0
    W = width
    H = width / phi

    pl = placement.lower()
    if pl == "center":
        x0 = -W * 0.5; y0 = -H * 0.5
    elif pl == "lowerleft":
        x0 = 0.0;      y0 = 0.0
    elif pl == "lowerright":
        x0 = -W;       y0 = 0.0
    elif pl == "upperleft":
        x0 = 0.0;      y0 = -H
    elif pl == "upperright":
        x0 = -W;       y0 = -H
    else:
        x0 = -W * 0.5; y0 = -H * 0.5

    # -----------------------------
    # Recursive subdivision (collect squares first)
    # -----------------------------
    # k cycles 0,1,2,3 for CCW peel sides: left, bottom, right, top.
    # For clockwise, we still use the same k progression, but later reverse arc direction.
    def _subdivide(rx, ry, rW, rH, k, depth, outSquares):
        if depth <= 0:
            return
        if rW <= tolerance or rH <= tolerance:
            return

        # Decide peel side by k, but ensure we peel the largest square consistent with rectangle aspect.
        # If rectangle is wide: square side = rH, remaining width = rW - rH.
        # If rectangle is tall: square side = rW, remaining height = rH - rW.
        wide = (rW >= rH)

        # Map k -> intended peel side
        # 0:left, 1:bottom, 2:right, 3:top
        if wide:
            s = rH
            if k == 0:      # left
                sx, sy = rx, ry
                nrx, nry = rx + s, ry
                nW, nH = rW - s, rH
            elif k == 2:    # right
                sx, sy = rx + (rW - s), ry
                nrx, nry = rx, ry
                nW, nH = rW - s, rH
            elif k == 1:    # bottom (fallback)
                sx, sy = rx, ry
                nrx, nry = rx, ry + s
                nW, nH = rW, rH - s
            else:           # top (fallback)
                sx, sy = rx, ry + (rH - s)
                nrx, nry = rx, ry
                nW, nH = rW, rH - s
        else:
            s = rW
            if k == 1:      # bottom
                sx, sy = rx, ry
                nrx, nry = rx, ry + s
                nW, nH = rW, rH - s
            elif k == 3:    # top
                sx, sy = rx, ry + (rH - s)
                nrx, nry = rx, ry
                nW, nH = rW, rH - s
            elif k == 0:    # left (fallback)
                sx, sy = rx, ry
                nrx, nry = rx + s, ry
                nW, nH = rW - s, rH
            else:           # right (fallback)
                sx, sy = rx + (rW - s), ry
                nrx, nry = rx, ry
                nW, nH = rW - s, rH

        # Store square + its k (we want arc orientation aligned with peel side)
        outSquares.append((sx, sy, s, k))

        # Next k rotates (CCW spiral)
        nk = (k + 1) % 4
        _subdivide(nrx, nry, nW, nH, nk, depth - 1, outSquares)

    squares = []
    _subdivide(float(x0), float(y0), float(W), float(H), 0, squareIterations, squares)
    if len(squares) == 0:
        return None

    # -----------------------------
    # Allocate arc segments across squares (global numSides)
    # -----------------------------
    # Weight by square size (larger squares get more segments), min 2 per arc.
    weights = [max(tolerance, s[2]) for s in squares]
    wsum = sum(weights)
    segs = []
    for w in weights:
        n = int(round(numSides * (w / wsum)))
        segs.append(max(2, n))

    # Normalize to exactly numSides while keeping >=2
    cur = sum(segs)
    while cur > numSides:
        j = max(range(len(segs)), key=lambda i: segs[i])
        if segs[j] > 2:
            segs[j] -= 1
            cur -= 1
        else:
            break
    while cur < numSides:
        j = max(range(len(segs)), key=lambda i: weights[i])
        segs[j] += 1
        cur += 1

    # -----------------------------
    # Build edges
    # -----------------------------
    sq_edges = []
    arc_edges = []

    for (sq, nseg) in zip(squares, segs):
        sx, sy, s, k = sq

        e, corners = _square_edges(sx, sy, s)
        sq_edges += e

        arc_edges += _arc_edges_from_square_corners(corners, k, nseg)

    all_edges = sq_edges + arc_edges
    wire = Wire.ByEdges(_as_topology(all_edges), tolerance=tolerance)
    if wire is None:
        return None

    # -----------------------------
    # Orient to direction + translate to origin
    # -----------------------------
    o = _safe_vertex(origin)

    d = _vec_norm(direction)
    z = [0.0, 0.0, 1.0]
    dot = max(-1.0, min(1.0, _vec_dot(z, d)))
    angle = math.degrees(math.acos(dot))
    axis = _vec_cross(z, d)
    ax_len = math.sqrt(axis[0]*axis[0] + axis[1]*axis[1] + axis[2]*axis[2])

    # Rotate about outer rectangle center
    cx = x0 + 0.5 * W
    cy = y0 + 0.5 * H
    cV = Vertex.ByCoordinates(_as_topology(_round(cx)), _as_topology(_round(cy)), 0.0)
    if ax_len > 1e-9 and angle > 1e-9:
        wire = Topology.Rotate(_as_topology(wire), _as_topology(cV), _as_topology(axis), _as_topology(angle))

    # Translate placement reference -> origin
    if pl == "center":
        refx, refy = cx, cy
    elif pl == "lowerleft":
        refx, refy = x0, y0
    elif pl == "lowerright":
        refx, refy = x0 + W, y0
    elif pl == "upperleft":
        refx, refy = x0, y0 + H
    elif pl == "upperright":
        refx, refy = x0 + W, y0 + H
    else:
        refx, refy = cx, cy

    refV = Vertex.ByCoordinates(_as_topology(_round(refx)), _as_topology(_round(refy)), 0.0)
    wire = Topology.Translate(_as_topology(wire), Vertex.X(_as_topology(o)) - Vertex.X(_as_topology(refV)), Vertex.Y(_as_topology(o)) - Vertex.Y(_as_topology(refV)), Vertex.Z(_as_topology(o)) - Vertex.Z(_as_topology(refV)))
    return wire


In [ ]:
@staticmethod
def GoldenSpiral(width: float = 1.0,
                 numSides: int = 96,
                 origin=None,
                 placement: str = "center",
                 direction: list = [0, 0, 1],
                 clockwise: bool = False,
                 squareIterations: int = 10,
                 edgeTypeKey: str = "edge_type",
                 squareEdgeValue: str = "square",
                 spiralEdgeValue: str = "spiral",
                 mantissa: int = 6,
                 tolerance: float = 0.0001,
                 silent: bool = False):
    """
    Creates the classic golden-rectangle subdivision squares + the “golden spiral”
    as quarter-circle arcs (approximated by segmented polylines).

    This implementation uses recursion to:
      - accept the next golden rectangle remainder,
      - add the next square,
      - add its quarter-circle arc,
      - recur.

    Edge tagging
    -----------
    Every edge is tagged by dictionary key `edgeTypeKey`:
      - square edges  -> `squareEdgeValue`
      - spiral edges  -> `spiralEdgeValue`

    Notes
    -----
    - The spiral here is the common *quarter-circle* construction (piecewise circular),
      not the true logarithmic spiral.
    - The arc orientation is “flipped along the diagonal of the square” by using the
      opposite corner as the arc center (same diagonal endpoints, mirrored arc).
    - Some Topologic kernels may drop/lose edge dictionaries when building a wire.
      This function *re-stamps* missing tags **before** any rotation/translation by
      classifying edges in local XY: axis-aligned edges are squares; others are spiral.
    """

    from topologicpy.Vertex import Vertex
    from topologicpy.Edge import Edge
    from topologicpy.Wire import Wire
    from topologicpy.Dictionary import Dictionary
    from topologicpy.Topology import Topology

    import math

    # -----------------------------
    # Helpers
    # -----------------------------
    def _safe_vertex(v):
        return v if v is not None else Vertex.Origin()

    def _round(x):
        return round(float(x), int(mantissa))

    def _vec_norm(v):
        l = math.sqrt(v[0]*v[0] + v[1]*v[1] + v[2]*v[2])
        if l < 1e-12:
            return [0.0, 0.0, 1.0]
        return [v[0]/l, v[1]/l, v[2]/l]

    def _vec_cross(a, b):
        return [a[1]*b[2]-a[2]*b[1], a[2]*b[0]-a[0]*b[2], a[0]*b[1]-a[1]*b[0]]

    def _vec_dot(a, b):
        return a[0]*b[0] + a[1]*b[1] + a[2]*b[2]

    def _get_edge_type(e):
        d = _record_dictionary(e)
        if d is None:
            return None
        return Dictionary.ValueAtKey(d, edgeTypeKey)

    def _set_edge_type(e, v):
        d = Dictionary.ByKeysValues([edgeTypeKey], [v])
        Topology.SetDictionary(e, d)
        return e

    def _edge_with_tag(v0, v1, tag_value):
        e = Edge.ByStartVertexEndVertex(_as_topology(v0), _as_topology(v1), tolerance=tolerance)
        return _set_edge_type(e, tag_value)

    def _dist_xy(a, b):
        dx = Vertex.X(_as_topology(a)) - Vertex.X(_as_topology(b))
        dy = Vertex.Y(_as_topology(a)) - Vertex.Y(_as_topology(b))
        return math.sqrt(dx*dx + dy*dy)

    def _square_edges_and_corners(sx, sy, s):
        bl = Vertex.ByCoordinates(_as_topology(_round(sx)), _as_topology(_round(sy)), 0.0)
        br = Vertex.ByCoordinates(_as_topology(_round(sx + s)), _as_topology(_round(sy)), 0.0)
        tr = Vertex.ByCoordinates(_as_topology(_round(sx + s)), _as_topology(_round(sy + s)), 0.0)
        tl = Vertex.ByCoordinates(_as_topology(_round(sx)), _as_topology(_round(sy + s)), 0.0)
        edges = [
            _edge_with_tag(bl, br, squareEdgeValue),
            _edge_with_tag(br, tr, squareEdgeValue),
            _edge_with_tag(tr, tl, squareEdgeValue),
            _edge_with_tag(tl, bl, squareEdgeValue),
        ]
        return edges, (bl, br, tr, tl)

    def _ang_from(center, p):
        return math.atan2(Vertex.Y(_as_topology(p)) - Vertex.Y(_as_topology(center)), Vertex.X(_as_topology(p)) - Vertex.X(_as_topology(center)))

    def _normalize_angle(a):
        # normalize to (-pi, pi]
        while a <= -math.pi:
            a += 2.0*math.pi
        while a > math.pi:
            a -= 2.0*math.pi
        return a

    def _arc_edges_flipped_diagonal(corners, k, nseg):
        """
        Build the quarter-circle inside a square, with the arc mirrored (“flipped”)
        across the diagonal chord.

        For each step k, the diagonal endpoints are the opposite corners; the arc
        center is one of the other two corners. Flipping swaps the center to the
        opposite corner of that pair.

        corners = (bl, br, tr, tl)

        k cycles 0,1,2,3 with the recursion (spiral progression).
        """
        bl, br, tr, tl = corners

        # diagonal endpoints + FLIPPED center
        if k == 0:
            p0, p1 = tl, br
            center = tr   # flipped (was bl)
        elif k == 1:
            p0, p1 = bl, tr
            center = tl   # flipped (was br)
        elif k == 2:
            p0, p1 = br, tl
            center = bl   # flipped (was tr)
        else:  # k == 3
            p0, p1 = tr, bl
            center = br   # flipped (was tl)

        r = max(tolerance, _dist_xy(center, p0))

        ang0 = _ang_from(center, p0)
        angT = _ang_from(center, p1)

        # choose +90 or -90 from ang0 that best matches angT
        candA = ang0 + math.pi/2.0
        candB = ang0 - math.pi/2.0
        dA = abs(_normalize_angle(candA - angT))
        dB = abs(_normalize_angle(candB - angT))
        ang1 = candA if dA <= dB else candB

        # reverse traversal if clockwise
        if clockwise:
            ang0, ang1 = ang1, ang0

        nseg = max(1, int(nseg))
        pts = []
        cx, cy = Vertex.X(_as_topology(center)), Vertex.Y(_as_topology(center))
        for i in range(nseg + 1):
            t = float(i) / float(nseg)
            a = ang0 + t * (ang1 - ang0)
            x = cx + r * math.cos(a)
            y = cy + r * math.sin(a)
            pts.append(Vertex.ByCoordinates(_as_topology(_round(x)), _as_topology(_round(y)), 0.0))

        edges = []
        for i in range(len(pts) - 1):
            edges.append(_edge_with_tag(pts[i], pts[i+1], spiralEdgeValue))
        return edges

    def _is_axis_aligned_local_xy(e):
        sv = Edge.StartVertex(_as_topology(e))
        ev = Edge.EndVertex(_as_topology(e))
        dx = abs(Vertex.X(_as_topology(ev)) - Vertex.X(_as_topology(sv)))
        dy = abs(Vertex.Y(_as_topology(ev)) - Vertex.Y(_as_topology(sv)))
        # In local XY (before any rotation), square edges are horizontal/vertical.
        # Use a slightly relaxed threshold to survive rounding/merging.
        eps = max(10.0*tolerance, 10.0**(-mantissa+1))
        return (dx <= eps) or (dy <= eps)

    # -----------------------------
    # Validate
    # -----------------------------
    try:
        width = float(width)
        if width <= 0:
            if not silent:
                print("Wire.GoldenSpiral: width must be > 0.")
            return None
        numSides = max(4, int(numSides))
        squareIterations = max(1, int(squareIterations))
    except Exception:
        if not silent:
            print("Wire.GoldenSpiral: invalid inputs.")
        return None

    # -----------------------------
    # Outer golden rectangle (local XY)
    # -----------------------------
    phi = (1.0 + math.sqrt(5.0)) / 2.0
    W = width
    H = width / phi

    pl = placement.lower()
    if pl == "center":
        x0, y0 = -W*0.5, -H*0.5
    elif pl == "lowerleft":
        x0, y0 = 0.0, 0.0
    elif pl == "lowerright":
        x0, y0 = -W, 0.0
    elif pl == "upperleft":
        x0, y0 = 0.0, -H
    elif pl == "upperright":
        x0, y0 = -W, -H
    else:
        x0, y0 = -W*0.5, -H*0.5

    # -----------------------------
    # Recursively collect squares: (sx, sy, s, k)
    # -----------------------------
    def _subdivide(rx, ry, rW, rH, k, depth, outSquares):
        if depth <= 0:
            return
        if rW <= tolerance or rH <= tolerance:
            return

        wide = (rW >= rH)

        # k corresponds to intended peel side in CCW progression:
        # 0:left, 1:bottom, 2:right, 3:top
        if wide:
            s = rH
            if k == 0:      # left
                sx, sy = rx, ry
                nrx, nry = rx + s, ry
                nW, nH = rW - s, rH
            elif k == 2:    # right
                sx, sy = rx + (rW - s), ry
                nrx, nry = rx, ry
                nW, nH = rW - s, rH
            elif k == 1:    # bottom (fallback)
                sx, sy = rx, ry
                nrx, nry = rx, ry + s
                nW, nH = rW, rH - s
            else:           # top (fallback)
                sx, sy = rx, ry + (rH - s)
                nrx, nry = rx, ry
                nW, nH = rW, rH - s
        else:
            s = rW
            if k == 1:      # bottom
                sx, sy = rx, ry
                nrx, nry = rx, ry + s
                nW, nH = rW, rH - s
            elif k == 3:    # top
                sx, sy = rx, ry + (rH - s)
                nrx, nry = rx, ry
                nW, nH = rW, rH - s
            elif k == 0:    # left (fallback)
                sx, sy = rx, ry
                nrx, nry = rx + s, ry
                nW, nH = rW - s, rH
            else:           # right (fallback)
                sx, sy = rx + (rW - s), ry
                nrx, nry = rx, ry
                nW, nH = rW - s, rH

        outSquares.append((sx, sy, s, k))
        _subdivide(nrx, nry, nW, nH, (k + 1) % 4, depth - 1, outSquares)

    squares = []
    _subdivide(float(x0), float(y0), float(W), float(H), 0, squareIterations, squares)
    if len(squares) == 0:
        return None

    # -----------------------------
    # Distribute global numSides across arcs (min 2 per arc)
    # -----------------------------
    weights = [max(tolerance, s) for (_, _, s, _) in squares]
    wsum = sum(weights)

    segs = []
    for w in weights:
        n = int(round(numSides * (w / wsum)))
        segs.append(max(2, n))

    cur = sum(segs)
    while cur > numSides:
        j = max(range(len(segs)), key=lambda i: segs[i])
        if segs[j] > 2:
            segs[j] -= 1
            cur -= 1
        else:
            break
    while cur < numSides:
        j = max(range(len(segs)), key=lambda i: weights[i])
        segs[j] += 1
        cur += 1

    # -----------------------------
    # Build geometry in local XY
    # -----------------------------
    sq_edges = []
    arc_edges = []

    for (sx, sy, s, k), nseg in zip(squares, segs):
        e, corners = _square_edges_and_corners(sx, sy, s)
        sq_edges += e
        arc_edges += _arc_edges_flipped_diagonal(corners, k, nseg)

    all_edges = sq_edges + arc_edges

    # Build wire (may merge edges internally)
    wire = Wire.ByEdges(_as_topology(all_edges), tolerance=tolerance)
    if wire is None:
        return None

    # -----------------------------
    # FIX: re-stamp missing edge tags (before rotation/translation!)
    # -----------------------------
    wEdges = Topology.Edges(_as_topology(wire)) or []
    for e in wEdges:
        if _get_edge_type(e) is not None:
            continue
        # classify in local XY: axis-aligned => square, else spiral
        _set_edge_type(e, squareEdgeValue if _is_axis_aligned_local_xy(e) else spiralEdgeValue)

    # -----------------------------
    # Orient to direction + translate to origin
    # -----------------------------
    o = _safe_vertex(origin)

    d = _vec_norm(direction)
    z = [0.0, 0.0, 1.0]
    dot = max(-1.0, min(1.0, _vec_dot(z, d)))
    ang_deg = math.degrees(math.acos(dot))
    axis = _vec_cross(z, d)
    ax_len = math.sqrt(axis[0]*axis[0] + axis[1]*axis[1] + axis[2]*axis[2])

    # rotate about outer rectangle center
    cx = x0 + 0.5 * W
    cy = y0 + 0.5 * H
    cV = Vertex.ByCoordinates(_as_topology(_round(cx)), _as_topology(_round(cy)), 0.0)
    if ax_len > 1e-9 and ang_deg > 1e-9:
        wire = Topology.Rotate(_as_topology(wire), _as_topology(cV), _as_topology(axis), _as_topology(ang_deg))

    # translate placement reference -> origin
    if pl == "center":
        refx, refy = cx, cy
    elif pl == "lowerleft":
        refx, refy = x0, y0
    elif pl == "lowerright":
        refx, refy = x0 + W, y0
    elif pl == "upperleft":
        refx, refy = x0, y0 + H
    elif pl == "upperright":
        refx, refy = x0 + W, y0 + H
    else:
        refx, refy = cx, cy

    refV = Vertex.ByCoordinates(_as_topology(_round(refx)), _as_topology(_round(refy)), 0.0)
    wire = Topology.Translate(_as_topology(wire), Vertex.X(_as_topology(o)) - Vertex.X(_as_topology(refV)), Vertex.Y(_as_topology(o)) - Vertex.Y(_as_topology(refV)), Vertex.Z(_as_topology(o)) - Vertex.Z(_as_topology(refV)))

    return wire


In [ ]:
gs = GoldenSpiral(width=10, numSides=30, squareIterations=10)
edges = Topology.Edges(_as_topology(gs))
square = []
spiral = []
for e in edges:
    d = _record_dictionary(e)
    t = Dictionary.ValueAtKey(d, "edge_type")
    if t == "square":
        square.append(e)
    elif t == "spiral":
        spiral.append(e)
    else:
        print(t)

Topology.Show(square, backgroundColor="white", projection="ortho", camera=[0,0,3])
Topology.Show(spiral, backgroundColor="white", projection="ortho", camera=[0,0,3])

In [ ]:
def generate_gmsh(face, mode="mesh", meshSize = None, tolerance = 0.0001, mantissa=6):
    """
    Creates a gmsh of triangular meshes from the input face.

    Parameters
    ----------
    face : topologic_core.Face
        The input face.
    meshSize : float , optional
        The desired mesh size.
    tolerance : float , optional
        The desired tolerance. Default is 0.0001.
    
    Returns
    -------
    topologic_core.Shell
        The shell of triangular meshes.

    """
    import os
    import warnings
    try:
        import numpy as np
    except:
        print("Face.Triangulate - Warning: Installing required numpy library.")
        try:
            os.system("pip install numpy")
        except:
            os.system("pip install numpy --user")
        try:
            import numpy as np
            print("Face.Triangulate - Warning: numpy library installed correctly.")
        except:
            warnings.warn("Face.Triangulate - Error: Could not import numpy. Please try to install numpy manually. Returning None.")
            return None
    try:
        import gmsh
    except:
        print("Face.Triangulate - Warning: Installing required gmsh library.")
        try:
            os.system("pip install gmsh")
        except:
            os.system("pip install gmsh --user")
        try:
            import gmsh
            print("Face.Triangulate - Warning: gmsh library installed correctly.")
        except:
            warnings.warn("Face.Triangulate - Error: Could not import gmsh. Please try to install gmsh manually. Returning None.")
            return None
    
    from topologicpy.Vertex import Vertex
    from topologicpy.Wire import Wire
    from topologicpy.Topology import Topology

    if not Topology.IsInstance(_as_topology(face), 'Face'):
        if not silent:
            print("Face.Triangulate - Error: The input face parameter is not a valid face. Returning None.")
        return None
    if not meshSize:
        bounding_face = Face.BoundingRectangle(face)
        bounding_face_vertices = Topology.Vertices(_as_topology(bounding_face))
        bounding_face_vertices_x = [Vertex.X(_as_topology(i), mantissa=mantissa) for i in bounding_face_vertices]
        bounding_face_vertices_y = [Vertex.Y(_as_topology(i), mantissa=mantissa) for i in bounding_face_vertices]
        width = max(bounding_face_vertices_x)-min(bounding_face_vertices_x)
        length = max(bounding_face_vertices_y)-min(bounding_face_vertices_y)
        meshSize = max([width,length])//10
    
    gmsh.initialize()
    face_external_boundary = Face.ExternalBoundary(face)
    external_vertices = Topology.Vertices(_as_topology(face_external_boundary))
    external_vertex_number = len(external_vertices)
    for i in range(external_vertex_number):
        gmsh.model.geo.addPoint(Vertex.X(_as_topology(external_vertices[i]), mantissa=mantissa), Vertex.Y(_as_topology(external_vertices[i]), mantissa=mantissa), Vertex.Z(_as_topology(external_vertices[i]), mantissa=mantissa), meshSize, i + 1)
    for i in range(external_vertex_number):
        if i < external_vertex_number-1:
            gmsh.model.geo.addLine(i+1, i+2, i+1)
        else:
            gmsh.model.geo.addLine(i+1, 1, i+1)
    gmsh.model.geo.addCurveLoop([i+1 for i in range(external_vertex_number)], 1)
    current_vertex_number = external_vertex_number
    current_edge_number = external_vertex_number
    current_wire_number = 1

    face_internal_boundaries = Face.InternalBoundaries(face)
    if face_internal_boundaries:
        internal_face_number = len(face_internal_boundaries)
        for i in range(internal_face_number):
            face_internal_boundary = face_internal_boundaries[i]
            internal_vertices = Topology.Vertices(_as_topology(face_internal_boundary))
            internal_vertex_number = len(internal_vertices)
            for j in range(internal_vertex_number):
                gmsh.model.geo.addPoint(Vertex.X(_as_topology(internal_vertices[j])), Vertex.Y(_as_topology(internal_vertices[j]), mantissa=mantissa), Vertex.Z(_as_topology(internal_vertices[j]), mantissa=mantissa), meshSize, current_vertex_number + j + 1)
            for j in range(internal_vertex_number):
                if j < internal_vertex_number-1:
                    gmsh.model.geo.addLine(current_vertex_number+j+1, current_vertex_number+j+2, current_edge_number+j+1)
                else:
                    gmsh.model.geo.addLine(current_vertex_number+j+1, current_vertex_number+1, current_edge_number+j+1)
            gmsh.model.geo.addCurveLoop([current_edge_number+i+1 for i in range(internal_vertex_number)], current_wire_number+1)
            current_vertex_number = current_vertex_number+internal_vertex_number
            current_edge_number = current_edge_number+internal_vertex_number
            current_wire_number = current_wire_number+1

    gmsh.model.geo.addPlaneSurface([i+1 for i in range(current_wire_number)])
    gmsh.model.geo.synchronize()
    if mode not in [1,3,5,6,7,8,9]:
        mode = 6
    gmsh.option.setNumber("Mesh.Algorithm", mode)
    gmsh.model.mesh.generate(2)         # For a 2D mesh
    nodeTags, nodeCoords, nodeParams = gmsh.model.mesh.getNodes(-1, -1)
    elemTypes, elemTags, elemNodeTags = gmsh.model.mesh.getElements(-1, -1)
    gmsh.write("C:/Users/sarwj/Downloads/viris_gallery.msh")
    gmsh.finalize()
    return gmsh

In [ ]:
from topologicpy.Face import Face
from topologicpy.Cell import Cell
w1 = Wire.Rectangle(width=10, length=8)
f1 = Face.ByWire(w1)
w1 = Wire.Rectangle(width=8, length=12)
f2 = Face.ByWire(w1)
f1 = Topology.Union(f1, f2)
f2 = Face.ByOffset(f1, offset=0.3)
f = Face.ByWires(Face.ExternalBoundary(f1), [Face.ExternalBoundary(f2)])
c2 = Cell.Prism(origin=Vertex.ByCoordinates(0,-2.5,0), width=0.2, length=2.75, height=4)
c3 = Cell.Prism(origin=Vertex.ByCoordinates(0,1,0), width=6, length=0.2, height=4)
g_face = Topology.Difference(f2, c2)
g_face = Topology.Difference(g_face, c3)
Topology.Show(g_face)

g_mesh = generate_gmsh(g_face, mode="mesh", meshSize = None, tolerance = 0.0001)
c = Cell.ByThickenedFace(f, bothSides=False, thickness=4)
c2 = Cell.Prism(origin=Vertex.ByCoordinates(0,-2.5,0), width=0.2, length=2.75, height=4, placement="bottom")
c3 = Cell.Prism(origin=Vertex.ByCoordinates(0,1,0), width=6, length=0.2, height=4, placement="bottom")
door = Cell.Prism(origin=Vertex.ByCoordinates(0,6,-0.2), width=2, length=1, height=2, placement="bottom")
c = Topology.Union(c, c2)
c = Topology.Union(c, c3)
c = Topology.Difference(c, door)
Topology.Show(c)
result = Topology.ExportToBREP(c, path=r"C:\Users\sarwj\Downloads\viris_gallery.brep")

In [ ]:
@staticmethod
def ShellGoldenRectangle(width: float = 1.0,
                         maxIterations: int = 10,
                         clockwise: bool = False,
                         origin=None,
                         placement: str = "center",
                         direction: list = [0, 0, 1],
                         mantissa: int = 6,
                         tolerance: float = 0.0001,
                         silent: bool = False):
    """
    Creates a golden rectangle (sized by `width`) and its recursive golden-rectangle
    subdivision squares (no spiral geometry).

    Output contains ONLY:
      - edges of all generated squares (tagged).

    Parameters
    ----------
    width : float
        The desired long side of the outer golden rectangle. Height is width/phi.
    maxIterations : int
        Number of subdivision squares to generate.
    clockwise : bool , optional
        Controls the square “peel” progression (affects which side each next square
        is taken from). Default is False.
    origin : topologic_core.Vertex, optional
        The location of the origin of the rectangle. Default is None which results in the rectangle being placed at (0, 0, 0).
    direction : list , optional
        The vector representing the up direction of the rectangle. Default is [0, 0, 1].
    placement : str , optional
        The description of the placement of the origin of the rectangle. This can be "center", "lowerleft", "upperleft", "lowerright", "upperright". It is case insensitive. Default is "center".
    mantissa : int , optional
        The desired length of the mantissa. Default is 6.
    tolerance : float , optional
        The desired tolerance. Default is 0.0001.
    silent : bool , optional
        If set to True, error and warning messages are suppressed. Default is False.

    Returns
    -------
    topologic_core.Shell
        A shell made from the faces of all subdivision squares (multiple loops).
    """

    from topologicpy.Face import Face
    from topologicpy.Topology import Topology
    import math

    # -----------------------------
    # Validate
    # -----------------------------
    width = float(width)
    if width <= 0:
        if not silent:
            print("Shell.GoldenRectangle - Error: width must be greater than 0. Returning None.")
        return None
    maxIterations = max(0, int(maxIterations))
    if maxIterations == 0:
        if silent:
            print("Shell.GoldenRectangle - Error: maxiterations must be larger than 0. Returning None.")
        return None
    if origin == None:
        origin = Vertex.Origin()
    if not Topology.IsInstance(_as_topology(origin), 'vertex'):
        if not silent:
            print("Shell.GoldenRectangle - Error: The input origin parameter is not a valid vertex. Returning None.")
        return None

    wire = GoldenRectangle(width=width,
                           maxIterations = maxIterations,
                           clockwise = clockwise,
                           origin=origin,
                           placement=placement,
                           direction = [0,0,1],
                           mantissa = mantissa,
                           tolerance = tolerance,
                           silent=silent)
    
    print(wire)
    
    phi = (1.0 + math.sqrt(5.0)) / 2.0
    W = width
    L = width / phi
    face = Face.Rectangle(width=W, length=L, placement=placement, origin=origin, direction=[0,0,1])
    Topology.Show(face, wire)
    shell = Topology.Slice(face, wire)
    faces = Topology.Faces(_as_topology(shell))

    if len(faces) < maxIterations+1:
        if not silent:
            print("Shell.GoldenRectangle - Warning: Could not create all the required faces. Consider increasing the size of the rectangle.")
    # -----------------------------
    # Orient to direction
    # -----------------------------
    if direction != [0, 0, 1]:
        shell = Topology.Orient(shell, origin=origin, dirA=[0, 0, 1], dirB=direction)
    return shell


gr = ShellGoldenRectangle(maxIterations=10, width=5, direction=[-1,0,0], placement="lowerleft")
vertices = Topology.Vertices(_as_topology(gr))
print("Vertices:",len(vertices))
edges = Topology.Edges(_as_topology(gr))
print("Edges:", len(edges))
faces = Topology.Faces(_as_topology(gr))
print("Faces:",len(faces))

Topology.Show(gr, backgroundColor="white", xAxis=True, yAxis=True, zAxis=True)

## New methods

In [ ]:
@staticmethod
def GoldenRectangle(width: float = 1.0,
                    maxIterations: int = 10,
                    clockwise: bool = False,
                    origin=None,
                    placement: str = "center",
                    direction: list = [0, 0, 1],
                    mantissa: int = 6,
                    tolerance: float = 0.0001,
                    silent: bool = False):
    """
    Creates a "golden rectangle". See https://en.wikipedia.org/wiki/Golden_rectangle.
    
    Parameters
    ----------
    width : float
        The desired long side of the outer golden rectangle. Height is width/phi.
    maxIterations : int
        Number of subdivision squares to generate.
    clockwise : bool , optional
        Controls the square “peel” progression (affects which side each next square
        is taken from). Default is False.
    origin : topologic_core.Vertex, optional
        The location of the origin of the rectangle. Default is None which results in the rectangle being placed at (0, 0, 0).
    direction : list , optional
        The vector representing the up direction of the rectangle. Default is [0, 0, 1].
    placement : str , optional
        The description of the placement of the origin of the rectangle. This can be "center", "lowerleft", "upperleft", "lowerright", "upperright". It is case insensitive. Default is "center".
    mantissa : int , optional
        The desired length of the mantissa. Default is 6.
    tolerance : float , optional
        The desired tolerance. Default is 0.0001.
    silent : bool , optional
        If set to True, error and warning messages are suppressed. Default is False.

    Returns
    -------
    topologic_core.Wire
        The created golden rectangle wire.
    """

    from topologicpy.Vertex import Vertex
    from topologicpy.Edge import Edge
    from topologicpy.Wire import Wire
    from topologicpy.Topology import Topology

    import math

    # -----------------------------
    # Helpers
    # -----------------------------
    def _safe_vertex(v):
        return v if v is not None else Vertex.Origin()

    def _round(x):
        return round(float(x), int(mantissa))

    def _edge(v0, v1):
        return Edge.ByStartVertexEndVertex(_as_topology(v0), _as_topology(v1), tolerance=tolerance, silent=silent)

    def _square_edges(sx, sy, s):
        bl = Vertex.ByCoordinates(_as_topology(_round(sx)), _as_topology(_round(sy)), 0.0)
        br = Vertex.ByCoordinates(_as_topology(_round(sx + s)), _as_topology(_round(sy)), 0.0)
        tr = Vertex.ByCoordinates(_as_topology(_round(sx + s)), _as_topology(_round(sy + s)), 0.0)
        tl = Vertex.ByCoordinates(_as_topology(_round(sx)), _as_topology(_round(sy + s)), 0.0)
        return [_edge(bl, br), _edge(br, tr), _edge(tr, tl), _edge(tl, bl)]

    # -----------------------------
    # Validate
    # -----------------------------
    width = float(width)
    if width <= 0:
        if not silent:
            print("Wire.GoldenRectangle - Error: width must be greater than 0. Returning None.")
        return None
    maxIterations = int(maxIterations)
    if maxIterations < 0:
        if not silent:
            print("Wire.GoldenRectangle - Error: maxIterations must be >= 0. Returning None.")
        return None
    clockwise = bool(clockwise)

    if origin == None:
        origin = Vertex.Origin()
    
    if not Topology.IsInstance(_as_topology(origin), 'vertex'):
        if not silent:
            print("Wire.GoldenRectangle - Error: The input origin parameter is not a valid vertex. Returning None.")
        return None
    
    placement = str(placement).lower()
    if not placement in ["center", "lowerleft", "lowerright", "upperleft", "upperright"]:
        if not silent:
            print("Wire.GoldenRectangle - Error: The input placement parameter is not a valid placement string. Returning None.")
        return None
    
    if not isinstance(direction, list):
        if not silent:
            print("Wire.GoldenRectangle - Error: The input direction parameter is not a valid list. Returning None.")
        return None
    
    direction = [x for x in direction if isinstance(x, (int, float))]
    
    if len(direction) != 3:
        if not silent:
            print("Wire.GoldenRectangle - Error: The input direction parameter is not a valid 3D vector. Returning None.")
        return None


    # -----------------------------
    # Canonical golden rectangle (UNIT width), centered at (0,0,0)
    # -----------------------------
    phi = (1.0 + math.sqrt(5.0)) / 2.0
    W0 = 1.0
    H0 = 1.0 / phi

    x0 = -W0 * 0.5
    y0 = -H0 * 0.5
    centerV = Vertex.ByCoordinates(0.0, 0.0, 0.0)

    # Outer boundary (canonical)
    boundary = Wire.Rectangle(origin=Vertex.ByCoordinates(_as_topology(_round(x0)), _as_topology(_round(y0)), 0.0), width=W0, length=H0, placement='lowerleft', direction=[0, 0, 1])

    # If no iterations requested, just return the boundary with final transforms
    if maxIterations == 0:
        wire = boundary
    else:
        # -----------------------------
        # Canonical recursive subdivision squares (k progression ALWAYS CCW canonical)
        # -----------------------------
        def _subdivide(rx, ry, rW, rH, k, depth, outSquares):
            if depth <= 0:
                return
            if rW <= tolerance or rH <= tolerance:
                if not silent:
                    print("Wire.GoldenRectangle - Warning: Edge lengths have fallen below tolerance. Stopping early.")
                return

            wide = (rW >= rH)

            # k: 0:left, 1:bottom, 2:right, 3:top  (canonical progression only)
            if wide:
                s = rH
                if k == 0:      # left
                    sx, sy = rx, ry
                    nrx, nry = rx + s, ry
                    nW, nH = rW - s, rH
                elif k == 2:    # right
                    sx, sy = rx + (rW - s), ry
                    nrx, nry = rx, ry
                    nW, nH = rW - s, rH
                elif k == 1:    # bottom (fallback)
                    sx, sy = rx, ry
                    nrx, nry = rx, ry + s
                    nW, nH = rW, rH - s
                else:           # top (fallback)
                    sx, sy = rx, ry + (rH - s)
                    nrx, nry = rx, ry
                    nW, nH = rW, rH - s
            else:
                s = rW
                if k == 1:      # bottom
                    sx, sy = rx, ry
                    nrx, nry = rx, ry + s
                    nW, nH = rW, rH - s
                elif k == 3:    # top
                    sx, sy = rx, ry + (rH - s)
                    nrx, nry = rx, ry
                    nW, nH = rW, rH - s
                elif k == 0:    # left (fallback)
                    sx, sy = rx, ry
                    nrx, nry = rx + s, ry
                    nW, nH = rW - s, rH
                else:           # right (fallback)
                    sx, sy = rx + (rW - s), ry
                    nrx, nry = rx, ry
                    nW, nH = rW - s, rH

            outSquares.append((sx, sy, s))
            _subdivide(nrx, nry, nW, nH, (k + 1) % 4, depth - 1, outSquares)

        squares = []
        _subdivide(float(x0), float(y0), float(W0), float(H0), 0, maxIterations, squares)
        if len(squares) == 0:
            if not silent:
                print("Wire.GoldenRectangle - Error: Could not create rectangle. Returning None.")
            return None

        # Build square edges (canonical)
        sq_edges = []
        for (sx, sy, s) in squares:
            e_list = _square_edges(sx, sy, s)
            if None in e_list:
                if not silent:
                    print("Wire.GoldenRectangle - Warning: Could not create an edge. Stopping early.")
                break
            sq_edges += e_list

        wire = Wire.ByEdges(_as_topology(sq_edges), tolerance=tolerance)
        wire = Topology.Merge(wire, boundary)

        if wire is None:
            if not silent:
                print("Wire.GoldenRectangle - Error: Could not create golden rectangle. Returning None.")
            return None

    # -----------------------------
    # FINAL transforms (only here)
    # -----------------------------

    # 1) Mirror (clockwise) about canonical center
    if clockwise:
        wire = Topology.Scale(_as_topology(wire), _as_topology(centerV), 1.0, -1.0, 1.0)

    # 2) Scale to requested width (canonical W0=1.0 => scale factors are (width, width, 1))
    wire = Topology.Scale(_as_topology(wire), _as_topology(centerV), _as_topology(width), _as_topology(width), 1.0)

    # 3) Translate so placement reference point lies at canonical origin (0,0,0)
    # After scaling:
    W = width
    H = width / phi
    pl = placement.lower()

    if pl == "center":
        refx, refy = 0.0, 0.0
    elif pl == "lowerleft":
        refx, refy = -W * 0.5, -H * 0.5
    elif pl == "lowerright":
        refx, refy =  W * 0.5, -H * 0.5
    elif pl == "upperleft":
        refx, refy = -W * 0.5,  H * 0.5
    elif pl == "upperright":
        refx, refy =  W * 0.5,  H * 0.5
    else:
        refx, refy = 0.0, 0.0

    wire = Topology.Translate(_as_topology(wire), -refx, -refy, 0.0)

    # 4) Orient/place (as requested)
    if direction != [0,0,1]:
        wire = Topology.Orient(wire, origin=origin, dirA=[0,0,1], dirB=direction)

    return wire


In [ ]:
@staticmethod
def GoldenSpiral(width: float = 1.0,
                 maxIterations: int = 10,
                 clockwise: bool = False,
                 sides: int = 96,
                 origin=None,
                 placement: str = "center",
                 direction: list = [0, 0, 1],
                 mantissa: int = 6,
                 tolerance: float = 0.0001,
                 silent: bool = False):
    """
    Creates a "golden spiral" as segmented quarter-circle arcs. See https://en.wikipedia.org/wiki/Golden_spiral
    
    Parameters
    ----------
    width : float
        The desired long side of the outer golden rectangle. Height is width/phi.
    maxIterations : int
        Number of subdivision squares to generate.
    clockwise : bool , optional
        Controls the square “peel” progression (affects which side each next square
        is taken from). Default is False.
    sides : int , optional
        The number of sides of the golden spiral (if included).
        Notes: If you set sides to be equal to maxIterations, you get the diagonals.
        It is best if the number of sides is a multiple of maxIterations.
        Default is 96.
    origin : topologic_core.Vertex, optional
        The location of the origin of the rectangle. Default is None which results in the rectangle being placed at (0, 0, 0).
    direction : list , optional
        The vector representing the up direction of the rectangle. Default is [0, 0, 1].
    placement : str , optional
        The description of the placement of the origin of the rectangle. This can be "center", "lowerleft", "upperleft", "lowerright", "upperright". It is case insensitive. Default is "center".
    mantissa : int , optional
        The desired length of the mantissa. Default is 6.
    tolerance : float , optional
        The desired tolerance. Default is 0.0001.
    silent : bool , optional
        If set to True, error and warning messages are suppressed. Default is False.

    Returns
    -------
    topologic_core.Wire
        The created golden spiral wire
    """

    from topologicpy.Vertex import Vertex
    from topologicpy.Edge import Edge
    from topologicpy.Wire import Wire
    from topologicpy.Topology import Topology
    from topologicpy.Cluster import Cluster

    import math

    # -----------------------------
    # Helpers
    # -----------------------------
    def _round(x):
        return round(float(x), int(mantissa))

    def _dist_xy(a, b):
        dx = Vertex.X(_as_topology(a)) - Vertex.X(_as_topology(b))
        dy = Vertex.Y(_as_topology(a)) - Vertex.Y(_as_topology(b))
        return math.sqrt(dx*dx + dy*dy)

    def _square_corners(sx, sy, s):
        bl = Vertex.ByCoordinates(_as_topology(_round(sx)), _as_topology(_round(sy)), 0.0)
        br = Vertex.ByCoordinates(_as_topology(_round(sx + s)), _as_topology(_round(sy)), 0.0)
        tr = Vertex.ByCoordinates(_as_topology(_round(sx + s)), _as_topology(_round(sy + s)), 0.0)
        tl = Vertex.ByCoordinates(_as_topology(_round(sx)), _as_topology(_round(sy + s)), 0.0)
        return (bl, br, tr, tl)

    def _ang_from(center, p):
        return math.atan2(Vertex.Y(_as_topology(p)) - Vertex.Y(_as_topology(center)), Vertex.X(_as_topology(p)) - Vertex.X(_as_topology(center)))

    def _normalize_angle(a):
        while a <= -math.pi:
            a += 2.0 * math.pi
        while a > math.pi:
            a -= 2.0 * math.pi
        return a

    def _arc_edges(center, p_start, p_end, nseg):
        """
        If nseg == 1: return the diagonal edge (p_start -> p_end).
        Else: return a polyline approximation of the quarter-circle.
        """
        nseg = max(1, int(nseg))
        if nseg == 1:
            return [Edge.ByStartVertexEndVertex(_as_topology(p_start), _as_topology(p_end), tolerance=tolerance)]

        r = max(tolerance, _dist_xy(center, p_start))
        ang0 = _ang_from(center, p_start)
        angT = _ang_from(center, p_end)

        # choose +/- 90 degrees from ang0 that best hits angT
        candA = ang0 + math.pi / 2.0
        candB = ang0 - math.pi / 2.0
        dA = abs(_normalize_angle(candA - angT))
        dB = abs(_normalize_angle(candB - angT))
        ang1 = candA if dA <= dB else candB

        cx, cy = Vertex.X(_as_topology(center)), Vertex.Y(_as_topology(center))
        pts = []
        for i in range(nseg + 1):
            t = float(i) / float(nseg)
            a = ang0 + t * (ang1 - ang0)
            x = cx + r * math.cos(a)
            y = cy + r * math.sin(a)
            pts.append(Vertex.ByCoordinates(_as_topology(_round(x)), _as_topology(_round(y)), 0.0))

        edges = []
        for i in range(len(pts) - 1):
            edges.append(Edge.ByStartVertexEndVertex(_as_topology(pts[i]), _as_topology(pts[i + 1]), tolerance=tolerance))
        return edges

    # -----------------------------
    # Validate
    # -----------------------------
    width = float(width)
    if width <= 0:
        if not silent:
            print("Wire.GoldenSpiral - Error: width must be greater than 0. Returning None.")
        return None
    maxIterations = int(maxIterations)
    if maxIterations <= 0:
        if not silent:
            print("Wire.GoldenSpiral - Error: maxIterations must be >= 0. Returning None.")
        return None
    
    sides = int(sides)
    if sides < maxIterations:
        if not silent:
            print("Wire.GoldenSpiral - Error: sides must be >= maxIterations. Returning None.")
        return None
    clockwise = bool(clockwise)

    if origin == None:
        origin = Vertex.Origin()
    
    if not Topology.IsInstance(_as_topology(origin), 'vertex'):
        if not silent:
            print("Wire.GoldenSpiral - Error: The input origin parameter is not a valid vertex. Returning None.")
        return None
    
    placement = str(placement).lower()
    if not placement in ["center", "lowerleft", "lowerright", "upperleft", "upperright"]:
        if not silent:
            print("Wire.GoldenSpiral - Error: The input placement parameter is not a valid placement string. Returning None.")
        return None
    
    if not isinstance(direction, list):
        if not silent:
            print("Wire.GoldenSpiral - Error: The input direction parameter is not a valid list. Returning None.")
        return None
    
    direction = [x for x in direction if isinstance(x, (int, float))]
    
    if len(direction) != 3:
        if not silent:
            print("Wire.GoldenSpiral - Error: The input direction parameter is not a valid 3D vector. Returning None.")
        return None

    # -----------------------------
    # Canonical golden rectangle (unit width), centered at (0,0,0)
    # -----------------------------
    phi = (1.0 + math.sqrt(5.0)) / 2.0
    W0 = 1.0
    H0 = 1.0 / phi

    x0 = -W0 * 0.5
    y0 = -H0 * 0.5
    centerV = Vertex.ByCoordinates(0.0, 0.0, 0.0)

    # -----------------------------
    # Canonical square sequence (CCW peel-side cycle)
    # -----------------------------
    # This is canonical; clockwise is applied later via mirroring.
    side_cycle = ["left", "bottom", "right", "top"]

    rx, ry, rW, rH = float(x0), float(y0), float(W0), float(H0)
    squares = []  # (sx, sy, s, side)

    for i in range(maxIterations):
        if rW <= tolerance or rH <= tolerance:
            break

        side = side_cycle[i % 4]

        if rW >= rH:
            s = rH
            if side == "right":
                sx, sy = rx + (rW - s), ry
                rW = rW - s
            else:  # left or fallback
                sx, sy = rx, ry
                rx = rx + s
                rW = rW - s
        else:
            s = rW
            if side == "top":
                sx, sy = rx, ry + (rH - s)
                rH = rH - s
            else:  # bottom or fallback
                sx, sy = rx, ry
                ry = ry + s
                rH = rH - s

        squares.append((sx, sy, s, side))

    if not squares:
        if not silent:
            print("Wire.GoldenSpiral - Error: Could not create square sequence. Returning None.")
        return None

    # -----------------------------
    # Distribute global sides across arcs (min 1 per arc)
    # -----------------------------
    weights = [max(tolerance, s) for (_, _, s, _) in squares]
    wsum = sum(weights) if sum(weights) > 0 else 1.0

    segs = []
    for w in weights:
        n = int(round(sides * (w / wsum)))
        segs.append(max(1, n))

    # normalize to exactly `sides` while maintaining >=1
    cur = sum(segs)
    while cur > sides:
        j = max(range(len(segs)), key=lambda i: segs[i])
        if segs[j] > 1:
            segs[j] -= 1
            cur -= 1
        else:
            break
    while cur < sides:
        j = max(range(len(segs)), key=lambda i: weights[i])
        segs[j] += 1
        cur += 1

    # -----------------------------
    # Build spiral arcs only (flipped-diagonal orientation by side)
    # -----------------------------
    spiral_edges = []
    last_end = None
    eps_join = 10.0 ** (-mantissa)

    for (sx, sy, s, side), nseg in zip(squares, segs):
        bl, br, tr, tl = _square_corners(sx, sy, s)

        # Flipped-diagonal mapping (by side)
        if side == "left":
            p0, p1 = tl, br
            c = tr
        elif side == "bottom":
            p0, p1 = bl, tr
            c = tl
        elif side == "right":
            p0, p1 = br, tl
            c = bl
        else:  # top
            p0, p1 = tr, bl
            c = br

        # continuity (swap endpoints if needed)
        if last_end is not None:
            if abs(Vertex.X(_as_topology(p0)) - Vertex.X(_as_topology(last_end))) > eps_join or abs(Vertex.Y(_as_topology(p0)) - Vertex.Y(_as_topology(last_end))) > eps_join:
                if abs(Vertex.X(_as_topology(p1)) - Vertex.X(_as_topology(last_end))) <= eps_join and abs(Vertex.Y(_as_topology(p1)) - Vertex.Y(_as_topology(last_end))) <= eps_join:
                    p0, p1 = p1, p0

        edges = _arc_edges(c, p0, p1, nseg)
        if edges:
            last_end = Edge.EndVertex(_as_topology(edges[-1]))
        spiral_edges += edges

    spiral = Wire.ByEdges(_as_topology(spiral_edges), tolerance=tolerance)
    if spiral is None:
        spiral = Topology.SelfMerge(Cluster.ByTopologies(_as_topology(spiral_edges)))
    if spiral is None:
        if not silent:
            print("Wire.GoldenSpiral - Error: Could not create spiral. Returning None.")
        return None

    # -----------------------------
    # FINAL transforms (only here)
    # -----------------------------

    # 1) Mirror for clockwise (negative scaling on Y axis about center)
    if clockwise:
        spiral = Topology.Scale(_as_topology(spiral), _as_topology(centerV), 1.0, -1.0, 1.0)

    # 2) Scale to requested width (canonical W0=1 => uniform XY scale = width)
    spiral = Topology.Scale(_as_topology(spiral), _as_topology(centerV), _as_topology(width), _as_topology(width), 1.0)

    # 3) Translate so placement reference of the *golden rectangle* lands at (0,0,0)
    W = width
    H = width / phi
    pl = placement.lower()

    if pl == "center":
        refx, refy = 0.0, 0.0
    elif pl == "lowerleft":
        refx, refy = -W * 0.5, -H * 0.5
    elif pl == "lowerright":
        refx, refy =  W * 0.5, -H * 0.5
    elif pl == "upperleft":
        refx, refy = -W * 0.5,  H * 0.5
    elif pl == "upperright":
        refx, refy =  W * 0.5,  H * 0.5
    else:
        refx, refy = 0.0, 0.0

    spiral = Topology.Translate(_as_topology(spiral), -refx, -refy, 0.0)

    # 4) Orient/place
    spiral = Topology.Orient(spiral, origin, [0, 0, 1], direction)

    return spiral


In [ ]:
@staticmethod
def ShellGoldenRectangle(width: float = 1.0,
                         maxIterations: int = 10,
                         clockwise: bool = False,
                         includeSpiral = True,
                         sides = 96,
                         origin=None,
                         placement: str = "center",
                         direction: list = [0, 0, 1],
                         mantissa: int = 6,
                         tolerance: float = 0.0001,
                         silent: bool = False):
    """
    Creates a "golden rectangle" with an optional "golden spiral". See https://en.wikipedia.org/wiki/Golden_rectangle and https://en.wikipedia.org/wiki/Golden_spiral.

    Parameters
    ----------
    width : float
        The desired long side of the outer golden rectangle. Height is width/phi.
    maxIterations : int
        Number of subdivision squares to generate.
    clockwise : bool , optional
        Controls the square “peel” progression (affects which side each next square
        is taken from). Default is False.
    includeSpiral : bool , optional
        If set to True, the golden spiral is included in the resulting shell. Default is True.
    sides : int , optional
        The number of sides of the golden spiral (if included).
        Notes: If you set sides to be equal to maxIterations, you get the diagonals.
        It is best if the number of sides is a multiple of maxIterations.
        Default is 96.
    origin : topologic_core.Vertex, optional
        The location of the origin of the rectangle. Default is None which results in the rectangle being placed at (0, 0, 0).
    direction : list , optional
        The vector representing the up direction of the rectangle. Default is [0, 0, 1].
    placement : str , optional
        The description of the placement of the origin of the rectangle. This can be "center", "lowerleft", "upperleft", "lowerright", "upperright". It is case insensitive. Default is "center".
    mantissa : int , optional
        The desired length of the mantissa. Default is 6.
    tolerance : float , optional
        The desired tolerance. Default is 0.0001.
    silent : bool , optional
        If set to True, error and warning messages are suppressed. Default is False.

    Returns
    -------
    topologic_core.Shell
        A shell made from the faces of all subdivision squares (multiple loops).
    """

    from topologicpy.Wire import Wire
    from topologicpy.Face import Face
    from topologicpy.Topology import Topology
    import math

    # -----------------------------
    # Validate
    # -----------------------------
    width = float(width)
    if width <= 0:
        if not silent:
            print("Shell.GoldenRectangle - Error: width must be greater than 0. Returning None.")
        return None
    maxIterations = int(maxIterations)
    if maxIterations <= 0:
        if not silent:
            print("Shell.GoldenRectangle - Error: maxIterations must be >= 0. Returning None.")
        return None
    if includeSpiral == True:
        sides = int(sides)
        if sides < maxIterations:
            if not silent:
                print("Shell.GoldenRectangle - Error: sides must be >= maxIterations. Returning None.")
            return None
    clockwise = bool(clockwise)
    if origin == None:
        origin = Vertex.Origin()
    
    if not Topology.IsInstance(_as_topology(origin), 'vertex'):
        if not silent:
            print("Shell.GoldenRectangle - Error: The input origin parameter is not a valid vertex. Returning None.")
        return None
    placement = str(placement).lower()
    if not placement in ["center", "lowerleft", "lowerright", "upperleft", "upperright"]:
        if not silent:
            print("Shell.GoldenRectangle - Error: The input placement parameter is not a valid placement string. Returning None.")
        return None
    if not isinstance(direction, list):
        if not silent:
            print("Shell.GoldenRectangle - Error: The input direction parameter is not a valid list. Returning None.")
        return None
    direction = [x for x in direction if isinstance(x, (int, float))]
    if len(direction) != 3:
        if not silent:
            print("Shell.GoldenRectangle - Error: The input direction parameter is not a valid 3D vector. Returning None.")
        return None

    gr = Wire.GoldenRectangle(width=width,
                           maxIterations = maxIterations,
                           clockwise = clockwise,
                           origin=origin,
                           placement=placement,
                           direction = [0,0,1],
                           mantissa = mantissa,
                           tolerance = tolerance,
                           silent=silent)
    
    if includeSpiral == True:
        gs = Wire.GoldenSpiral(width=width,
                           maxIterations = maxIterations,
                           clockwise = clockwise,
                           sides = sides,
                           origin=origin,
                           placement=placement,
                           direction = [0,0,1],
                           mantissa = mantissa,
                           tolerance = tolerance,
                           silent=silent)
        gr = Topology.Merge(gr, gs)

    
    
    phi = (1.0 + math.sqrt(5.0)) / 2.0
    W = width
    L = width / phi
    face = Face.Rectangle(width=W, length=L, placement=placement, origin=origin, direction=[0,0,1])
    shell = Topology.Slice(face, gr)
    faces = Topology.Faces(_as_topology(shell))

    if len(faces) < maxIterations+1:
        if not silent:
            print("Shell.GoldenRectangle - Warning: Could not create all the required faces. Consider increasing the size of the rectangle.")
    # -----------------------------
    # Orient to direction
    # -----------------------------
    if direction != [0, 0, 1]:
        shell = Topology.Orient(shell, origin=origin, dirA=[0, 0, 1], dirB=direction)
    return shell

In [ ]:
# gs1 = GoldenSpiral(maxIterations=4, sides=4, clockwise=True, placement="lowerright", direction=[1,0,0])
# gs2 = GoldenSpiral(maxIterations=4, sides=4, clockwise=False, placement="lowerright", )
# rect1 = GoldenRectangle(maxIterations=4, clockwise=True, placement="lowerright", direction=[1,0,0])
# rect2 = GoldenRectangle(maxIterations=4, clockwise=False, placement="lowerright")
shell1 = Shell.GoldenRectangle(maxIterations=4, clockwise=False, includeSpiral=True, sides=8, placement="lowerright")
print(shell1)
faces = Topology.Faces(_as_topology(shell1))
print(len(faces))
Topology.Show(shell1, backgroundColor="white", xAxis=True, yAxis=True, zAxis=True)
# Topology.Show(gs1, rect1, backgroundColor="white", xAxis=True, yAxis=True, zAxis=True)

# Topology.Show(gs2, rect2, backgroundColor="white", xAxis=True, yAxis=True, zAxis=True)